# 🎓 Working with Parcels output

This tutorial covers the format of the trajectory output exported by Parcels. **Parcels does not include advanced analysis or plotting functionality**, which users are suggested to write themselves to suit their research goals. Here we provide some starting points to explore the parcels output files yourself.

- [**Reading the output file**](#reading-the-output-file)
- [**Trajectory data structure**](#trajectory-data-structure)
- [**Analysis**](#analysis)
- [**Plotting**](#plotting)
- [**Animations**](#animations)

For more advanced reading and tutorials on the analysis of Lagrangian trajectories, we recommend checking out the [Lagrangian Diagnostics Analysis Cookbook](https://lagrangian-diags.readthedocs.io/en/latest/tutorials.html) and the project in general.

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

import parcels
import parcels.tutorial

First we need to create some parcels output to analyze. We simulate a set of particles using the setup described in the [Delay start tutorial](https://docs.oceanparcels.org/en/latest/examples/tutorial_delaystart.html). We will also add some user defined metadata to the output file.

In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)
ds_fields.load()  # load the dataset into memory

# Convert to SGRID-compliant dataset and create FieldSet
fields = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

In [ ]:
# Particle locations and initial time
npart = 10  # number of particles to be released
lon = 32 * np.ones(npart)
lat = np.linspace(-32.5, -30.5, npart, dtype=np.float32)
time = ds_fields.time.values[0] + np.arange(0, npart) * np.timedelta64(2, "h")
z = np.repeat(ds_fields.depth.values[0], npart)

pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, lon=lon, lat=lat, time=time, z=z
)

output_file = parcels.ParticleFile("output.parquet", outputdt=np.timedelta64(2, "h"))

Parcels saves some metadata in the output file with every simulation (Parcels version, CF convention information, etc.). This metadata is just a dictionary which is propogated to the parquet metadata. We are free to manipulate this dictionary to add any custom metadata relevant to our simulation. Here we add a custom metadata field `date_created` to the output file.

In [ ]:
output_file.metadata["date_created"] = datetime.now().isoformat()
output_file.metadata

To write the metadata to the output_file, we need to add it before running `pset.execute()` which writes the particleset including the metadata to the output_file.

In [ ]:
pset.execute(
    parcels.kernels.AdvectionRK2,
    runtime=np.timedelta64(48, "h"),
    dt=np.timedelta64(5, "m"),
    output_file=output_file,
)

## Reading the output file



```{note}
As of Parcels v4, the default output format is [`parquet`](https://parquet.apache.org/) (instead of `zarr`). The `parquet` output format is a tabular format, in which every row corresponds to an observation of a particle trajectory. The `zarr` output format is a multidimensional array format, in which the data is stored in a 2D array with dimensions `traj` and `obs`. The `parquet` format is more compact and faster to read.

However, the `parquet` format does not support the [CF-convention for trajectories data](http://cfconventions.org/cf-conventions/v1.6.0/cf-conventions.html#_multidimensional_array_representation_of_trajectories) implemented with the [NCEI trajectory template](https://www.ncei.noaa.gov/data/oceans/ncei/formats/netcdf/v2.0/trajectoryIncomplete.cdl). We are working on efficient tooling to convert the parcels `parquet` output into a CF-compliant format.

TODO: Add link to tracking issue on github for this tooling.
```

Parcels exports output trajectories in `parquet` [format](https://parquet.apache.org/). Files in `parquet` are stored in tabular data, so each row corresponds to a particle at a given time step, and columns correspond to particle attributes (lon, lat, time, etc.). 

The files can be analysed with a wide range of tools, including `pandas`, `polars` or the [`lt toolbox`](https://github.com/oj-tooth/lt_toolbox). The latter is specifically designed for the analysis of Lagrangian trajectories, and can be used to compute a wide range of Lagrangian diagnostics- but is still in alpha stage of development.

In Polars, these files can be opened with `polars.read_parquet`:

In [ ]:
df_polars = pl.read_parquet("output.parquet")

print(df_polars)

To see our custom metadata field `date_created`, we need to use `pyarrow.parquet`

In [ ]:
schema = pq.read_schema("output.parquet")
for k, v in schema.metadata.items():
    print(k.decode(), "->", v.decode())

As you may have noticed above, the `time` is shown as a `float64` (in seconds) in `df_polars`. That is because `polars.read_parquet` does not automatically convert the cftime. To handle this, we also provide a helper function `parcels.read_particlefile` to read ParticleFiles, which does automatically convert the cftime. 

In [ ]:
df_particles = parcels.read_particlefile("output.parquet")

print(df_particles)

## Trajectory data structure

The output dataset used here contains **10 particles** and **25 observations**. Not every particle has 25 observations however; since we released particles at different times some particle trajectories are shorter than others.

In [ ]:
for traj in df_particles.partition_by("particle_id", maintain_order=True):
    time_origin = pd.Timestamp(fieldset.time_interval.left).to_pydatetime()
    time_in_hour = (traj["time"] - time_origin).dt.total_hours()
    traj_id = traj["particle_id"][0]
    print(f"Particle {traj_id}: " + "".join(f"{int(t):2d}  " for t in time_in_hour))

Note how the first observation occurs at a different time for each trajectory.


## Plotting trajectories

Parcels output consists of particle trajectories through time and space. An important way to explore patterns in this information is to draw the trajectories in space. The [**trajan**](https://opendrift.github.io/trajan/index.html) package can be used to quickly plot parcels results, but users are encouraged to create their own figures, for example by using the comprehensive [**matplotlib**](https://matplotlib.org/) library. Here we show a basic setup on how to process the parcels output into trajectory plots and animations.

```{warning}
Trajan is not yet compatible with the `parquet` output format, but we are working on a solution to this.
```

Since the `parquet` output format is tabular, a simple plot of the longitude and latitude of all particles will show one continuous line:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(df_particles["lon"], df_particles["lat"], ".-")
plt.show()

To show the trajectories of individual particles, we must group the data by trajectory and plot each trajectory separately:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for traj in df_particles.partition_by("particle_id", maintain_order=True):
    traj_id = traj["particle_id"][0]
    ax.plot(traj["lon"], traj["lat"], ".-", label=f"P{traj_id}")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)
plt.tight_layout()
plt.show()

However, if you want to plot only the locations at a certain time step, you can simply select the data at that time step:

In [ ]:
time_step = np.timedelta64(18, "h")
time_to_plot = fieldset.time_interval.left + time_step
particles = df_particles.filter(pl.col("time") == pl.lit(time_to_plot))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(particles["lon"], particles["lat"], "o")
title_time = pd.to_datetime(time_to_plot).strftime("%Y-%m-%d %H:%M:%S")
ax.set_title(f"Particle locations at {title_time}")
plt.show()

Or, if you want to plot the particles a certain amount of time after they were released, you can select the data based on the time since release:

In [ ]:
time_step = np.timedelta64(18, "h")
particles = df_particles.filter(
    (pl.col("time") - pl.col("time").min().over("particle_id")) == pl.lit(time_step)
)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(particles["lon"], particles["lat"], "o")
ax.set_title(f"Particle locations {time_step} after their release")
plt.show()

We can plot the distance travelled as a function of absolute and relative time:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

for traj in df_particles.partition_by("particle_id", maintain_order=True):
    distance = np.sqrt(
        (traj["lon"] - traj["lon"][0]) ** 2 + (traj["lat"] - traj["lat"][0]) ** 2
    )
    ax[0].plot(traj["time"], distance, ".-", label=f"P{traj['particle_id'][0]}")

    rel_time = (traj["time"] - traj["time"][0]).dt.total_hours()
    ax[1].plot(rel_time, distance, ".-", label=f"P{traj['particle_id'][0]}")

ax[0].set_xlabel("Date")
ax[0].set_ylabel("Distance travelled [degrees]")
ax[0].tick_params(axis="x", labelrotation=45)
ax[1].set_xlabel("Time since release [hours]")
ax[1].set_ylabel("Distance travelled [degrees]")

plt.tight_layout()
plt.show()

### Animations


Trajectory plots like the ones above can become very cluttered for large sets of particles. To better see patterns, it's a good idea to create an animation in time and space. To do this, matplotlib offers an [animation package](https://matplotlib.org/stable/api/animation_api.html). Here we show how to use the [**FuncAnimation**](https://matplotlib.org/3.3.2/api/_as_gen/matplotlib.animation.FuncAnimation.html#matplotlib.animation.FuncAnimation) class to animate parcels trajectory data.


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib
from matplotlib.animation import FuncAnimation

# for interactive display of animation
plt.rcParams["animation.html"] = "jshtml"

In [ ]:
time_step = np.timedelta64(2, "h")  # time step for animation frames

timerange = np.arange(
    np.nanmin(df_particles["time"]),
    np.nanmax(df_particles["time"]) + time_step,
    time_step,
)

# set up a unique color for each trajectory
colormap = matplotlib.colormaps["tab20b"]
trajectory_to_color = {}
for i, trajectory in enumerate(df_particles["particle_id"].unique()):
    trajectory_to_color[trajectory] = colormap(
        i / max(len(df_particles["particle_id"].unique()) - 1, 1)
    )

# figure setup
fig, ax = plt.subplots(figsize=(6, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_xlim(30, 33)
ax.set_xticks(np.arange(30, 33.5, 0.5))
ax.set_xlabel("Longitude (deg E)")
ax.set_ylim(-33, -30)
ax.set_yticks(ticks=np.arange(-33, -29.5, 0.5))
ax.set_yticklabels(np.arange(33, 29.5, -0.5).astype(str))
ax.set_ylabel("Latitude (deg S)")
ax.coastlines()
ax.add_feature(cfeature.LAND)

# --> plot first timestep
particles = df_particles.filter(pl.col("time") == pl.lit(timerange[0]))
scatter = ax.scatter(
    particles["lon"],
    particles["lat"],
    s=10,
    c=[trajectory_to_color[p] for p in particles["particle_id"]],
)

# --> initialize trails
trail_plot = []

# Set initial title
t_str = pd.to_datetime(timerange[0]).strftime(
    "%Y-%m-%d %H:%M:%S"
)  # Format datetime nicely
title = ax.set_title(f"Particles at t = {t_str}")


# loop over for animation
def animate(i):
    t_str = pd.to_datetime(timerange[i]).strftime("%Y-%m-%d %H:%M:%S")
    title.set_text(f"Particles at t = {t_str}")

    # Find particles at current time
    particles = df_particles.filter(pl.col("time") == pl.lit(timerange[i]))

    if len(particles) > 0:
        scatter.set_offsets(np.c_[particles["lon"], particles["lat"]])
        scatter.set_color([trajectory_to_color[p] for p in particles["particle_id"]])

        # --> reset trails
        for trail in trail_plot:
            trail.remove()
        trail_plot.clear()
        trail_length = min(10, i)  # trails will have max length of 10 time steps
        if trail_length > 0:
            for traj in particles["particle_id"].unique():
                traj_trail = df_particles.filter(
                    (pl.col("particle_id") == traj)
                    & (pl.col("time") >= pl.lit(timerange[max(0, i - trail_length)]))
                    & (pl.col("time") <= pl.lit(timerange[i]))
                )
                if len(traj_trail) > 1:
                    (trail,) = ax.plot(
                        traj_trail["lon"],
                        traj_trail["lat"],
                        color=trajectory_to_color[traj],
                        linewidth=0.6,
                        alpha=0.6,
                    )
                    trail_plot.append(trail)
    else:
        scatter.set_offsets(np.empty((0, 2)))


# Create animation
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim